# CIFAR-10: exploring, clustering and classifying
Setup, data loading and the train, validation and test samples used in every part.

In [ ]:
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torchvision import datasets

In [ ]:
SEED = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

In [ ]:
train_set = datasets.CIFAR10("data", train=True, download=True)
test_set = datasets.CIFAR10("data", train=False, download=True)

train_images, train_labels = train_set.data, np.array(train_set.targets)
test_images, test_labels = test_set.data, np.array(test_set.targets)
classes = train_set.classes
print(train_images.shape, test_images.shape)

In [ ]:
sns.set_theme(style="white", context="paper", font_scale=0.9)
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight"})
class_colours = dict(zip(classes, sns.color_palette("tab10", len(classes))))

## Samples
Stratified random samples with the seed: 500 training and 50 validation images per class from the training set, and 50 test images per class from the test set.

In [ ]:
rng = np.random.default_rng(SEED)

def sample_per_class(labels, sizes):
    # one shuffle per class, sliced in order, so the samples cannot overlap
    samples = [[] for _ in sizes]
    for c in range(len(classes)):
        idx = rng.permutation(np.flatnonzero(labels == c))
        start = 0
        for sample, n in zip(samples, sizes):
            sample.append(idx[start:start + n])
            start += n
    return [np.sort(np.concatenate(s)) for s in samples]

train_idx, val_idx = sample_per_class(train_labels, [500, 50])
(test_idx,) = sample_per_class(test_labels, [50])

for name, idx, labels in [("train", train_idx, train_labels),
                          ("val", val_idx, train_labels),
                          ("test", test_idx, test_labels)]:
    print(name, len(idx), np.bincount(labels[idx]))
print("train and val overlap:", len(np.intersect1d(train_idx, val_idx)))